# Detectives del Código

Cada ejercicio viola un principio SOLID

En equipo:
1. Lee el código y ejecútalo.
2. Responde: ¿qué principio se viola? 
3. ¿Por qué? : señala qué parte del código lo causa y explica con tus propiar palabras por que.



## Ejercicio 01: Cancelación de Pólizas

`PolizaColectiva` hereda de `Poliza`, pero no permite hacer lo mismo que su clase base.


In [ ]:
class Poliza:
    def __init__(self, suma_asegurada):
        self.suma_asegurada = suma_asegurada
        self.activa = True

    def cancelar(self):
        self.activa = False
        return "Póliza cancelada"


class PolizaColectiva(Poliza):
    def cancelar(self):
        raise Exception("Las pólizas colectivas no se cancelan individualmente")


def procesar_cancelacion(poliza):
    """Función genérica: no debería importar de qué tipo es la póliza."""
    resultado = poliza.cancelar()
    print(resultado)

#Ejecucion
procesar_cancelacion(Poliza(500000))
try:
    procesar_cancelacion(PolizaColectiva(2000000))
except Exception as e:
    print(f"Error inesperado: {e}")


Póliza cancelada
Error inesperado: Las pólizas colectivas no se cancelan individualmente


**Pregunta:** si `PolizaColectiva` es un tipo de `Poliza`, ¿debería poder usarse en cualquier lugar donde se use una `Poliza`?

<details>
<summary><b>Ver respuesta</b></summary>

**Principio violado: L (Sustitución de Liskov)**

`PolizaColectiva` hereda de `Poliza` pero no puede hacer lo que su clase base promete (`cancelar`). Cualquier código que trate a todas las pólizas por igual se rompe al recibir una `PolizaColectiva`.

**Corrección:** no heredar de `Poliza` si no se puede cumplir su comportamiento. `PolizaColectiva` puede manejar su propio proceso de baja, sin el método `cancelar` individual.
</details>


## Ejercicio 02: Registro de Siniestros

El registrador de siniestros está atado directamente a una sola forma de guardar la información: un archivo de texto.


In [ ]:
class ArchivoTexto:
    def guardar(self, registro):
        with open("siniestros.txt", "a", encoding="utf-8") as f:
            f.write(registro + "\n")


class RegistradorSiniestro:
    def __init__(self):
        self.almacen = ArchivoTexto()  # atado a una implementación concreta

    def registrar(self, siniestro):
        self.almacen.guardar(siniestro)


# Ejecucion
registrador = RegistradorSiniestro()
registrador.registrar("Siniestro #4521 - Auto - $35,000")

with open("siniestros.txt", encoding="utf-8") as f:
    print("Contenido del archivo:")
    print(f.read())


Contenido del archivo:
Siniestro #4521 - Auto - $35,000



**Pregunta:** si mañana la aseguradora decide guardar los siniestros en un archivo CSV en vez de texto plano, ¿qué clases hay que modificar?

<details>
<summary><b>Ver respuesta</b></summary>

**Principio violado: D (Inversión de Dependencias)**

`RegistradorSiniestro` crea directamente un `ArchivoTexto` dentro de su propio constructor. Si cambia la forma de guardar los siniestros (CSV, Excel, otro archivo), hay que modificar `RegistradorSiniestro`, aunque su trabajo (registrar el siniestro) no cambió en absoluto.

**Corrección:** recibir el almacén como parámetro (inyección de dependencia), en vez de crearlo adentro.
</details>


In [ ]:
# Solución corregida
import csv

class ArchivoCSV:
    def guardar(self, registro):
        with open("siniestros.csv", "a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow([registro])


class RegistradorSiniestroV2:
    def __init__(self, almacen):
        self.almacen = almacen  # se recibe desde afuera

    def registrar(self, siniestro):
        self.almacen.guardar(siniestro)


# Ahora cambiar la forma de guardar no toca RegistradorSiniestroV2
RegistradorSiniestroV2(ArchivoTexto()).registrar("Siniestro #4522 - Vida - $500,000")
RegistradorSiniestroV2(ArchivoCSV()).registrar("Siniestro #4523 - Gastos médicos - $12,000")

with open("siniestros.csv", encoding="utf-8") as f:
    print("Contenido del CSV:")
    print(f.read())


Contenido del CSV:
"Siniestro #4523 - Gastos médicos - $12,000"



## Ejercicio 03: Emisión de Póliza

Una sola clase calcula la prima, genera el documento de la póliza y notifica al asegurado.


In [ ]:
class Poliza:
    def __init__(self, suma_asegurada, tasa):
        self.suma_asegurada = suma_asegurada
        self.tasa = tasa

    def calcular_prima(self):
        return self.suma_asegurada * self.tasa

    def generar_documento_pdf(self):
        print(f"Generando carátula de póliza. Prima anual: ${self.calcular_prima():.2f}")

    def notificar_asegurado(self, email):
        print(f"Enviando póliza a {email}")


poliza = Poliza(suma_asegurada=500000, tasa=0.012)
poliza.generar_documento_pdf()
poliza.notificar_asegurado("asegurado@correo.com")


Generando carátula de póliza. Prima anual: $6000.00
Enviando póliza a asegurado@correo.com


**Pregunta:** si mañana cambia el formato del documento de póliza, o el proveedor de correo, ¿cuántas razones distintas tiene esta clase para cambiar?

<details>
<summary><b>Ver respuesta</b></summary>

**Principio violado: S (Responsabilidad Única)**

`Poliza` calcula la prima, genera el documento y notifica al asegurado: son 3 trabajos distintos en una sola clase. Cambiar el cálculo actuarial de la prima, el formato del documento, o el proveedor de correo son razones de cambio completamente independientes.

**Corrección:** separar en `Poliza` (solo datos y cálculo de prima), `GeneradorDocumento` y `Notificador`.
</details>


In [ ]:
# Solución corregida
class PolizaV2:
    def __init__(self, suma_asegurada, tasa):
        self.suma_asegurada = suma_asegurada
        self.tasa = tasa

    def calcular_prima(self):
        return self.suma_asegurada * self.tasa


class GeneradorDocumento:
    def generar(self, poliza):
        print(f"Generando carátula de póliza. Prima anual: ${poliza.calcular_prima():.2f}")


class NotificadorAsegurado:
    def enviar(self, email):
        print(f"Enviando póliza a {email}")


poliza_v2 = PolizaV2(500000, 0.012)
GeneradorDocumento().generar(poliza_v2)
NotificadorAsegurado().enviar("asegurado@correo.com")


Generando carátula de póliza. Prima anual: $6000.00
Enviando póliza a asegurado@correo.com


## Ejercicio 04: Reserva y Beneficio por Fallecimiento

Una sola clase base obliga a todos los productos a tener los mismos métodos, aunque no les apliquen.


In [ ]:
class ProductoActuarial:
    def calcular_reserva(self):
        raise NotImplementedError

    def pagar_beneficio_fallecimiento(self):
        raise NotImplementedError


class SeguroVida(ProductoActuarial):
    def calcular_reserva(self):
        return 45000.0

    def pagar_beneficio_fallecimiento(self):
        return 500000.0


class SeguroAuto(ProductoActuarial):
    def calcular_reserva(self):
        return 8000.0

    def pagar_beneficio_fallecimiento(self):
        return 0.0  # un seguro de auto no paga beneficio por fallecimiento


# Demo
productos = [SeguroVida(), SeguroAuto()]
for p in productos:
    print(type(p).__name__, "reserva:", p.calcular_reserva(),
          "| beneficio fallecimiento:", p.pagar_beneficio_fallecimiento())


SeguroVida reserva: 45000.0 | beneficio fallecimiento: 500000.0
SeguroAuto reserva: 8000.0 | beneficio fallecimiento: 0.0


**Pregunta:** ¿por qué `SeguroAuto` necesita un método `pagar_beneficio_fallecimiento()` que siempre devuelve 0?

<details>
<summary><b>Ver respuesta</b></summary>

**Principio violado: I (Segregación de Interfaces)**

`ProductoActuarial` obliga a implementar `calcular_reserva()` y `pagar_beneficio_fallecimiento()` aunque no siempre apliquen. Esto genera métodos "de relleno" sin sentido real (`return 0.0` porque no aplica al producto).

**Corrección:** dividir en dos clases base pequeñas — `TieneReserva` y `PagaBeneficioFallecimiento` — y que cada producto implemente solo la que le corresponde.
</details>


---

## Ejercicio 05: Prima según Tipo de Seguro

Una función calcula la prima según el tipo de seguro, usando `if/elif`.


In [ ]:
def calcular_prima(tipo_seguro, suma_asegurada):
    if tipo_seguro == "vida":
        return suma_asegurada * 0.012
    elif tipo_seguro == "auto":
        return suma_asegurada * 0.035
    else:
        return suma_asegurada * 0.012  # por defecto, se cotiza como vida


# Demo
print("Vida:", calcular_prima("vida", 500000))
print("Auto:", calcular_prima("auto", 200000))
print("Gastos médicos (no existe todavía):", calcular_prima("gastos_medicos", 300000))  # se cotiza mal, sin avisar


Vida: 6000.0
Auto: 7000.000000000001
Gastos médicos (no existe todavía): 3600.0


**Pregunta:** si la aseguradora lanza un producto de "gastos médicos" con su propia tasa, ¿qué parte del código hay que tocar?

<details>
<summary><b>Ver respuesta</b></summary>

**Principio violado: O (Abierto/Cerrado)**

Cada producto nuevo obliga a **modificar** la función `calcular_prima`, agregando otro `elif`. Si se te olvida (como con `"gastos_medicos"` en la demo), el producto se cotiza mal en silencio — un error grave en un contexto actuarial.

**Corrección:** usar una función o clase distinta por producto, y elegir cuál usar sin tocar el código que ya funciona.
</details>


In [ ]:
# Solución corregida
def prima_vida(suma_asegurada):
    return suma_asegurada * 0.012

def prima_auto(suma_asegurada):
    return suma_asegurada * 0.035

def prima_gastos_medicos(suma_asegurada):
    return suma_asegurada * 0.02

tasas_por_producto = {
    "vida": prima_vida,
    "auto": prima_auto,
    "gastos_medicos": prima_gastos_medicos,
}

def calcular_prima_v2(tipo_seguro, suma_asegurada):
    funcion = tasas_por_producto[tipo_seguro]
    return funcion(suma_asegurada)

print("Gastos médicos corregido:", calcular_prima_v2("gastos_medicos", 300000))
# Agregar un producto nuevo solo requiere agregar una entrada al diccionario


Gastos médicos corregido: 6000.0


## Respuestas

| Ejercicios | Tema | Principio violado |
|---|---|---|
| 1 | Cancelación de Pólizas | L |
| 2 | Registro de Siniestros | D |
| 3 | Emisión de Póliza | S |
| 4 | Reserva y Beneficio por Fallecimiento | I |
| 5 | Prima según Tipo de Seguro | O |

